In [14]:
import os

from dotenv import load_dotenv
from IPython.display import display, Markdown

from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient

In [15]:
load_dotenv()

# llm = ChatGroq(
#     model="openai/gpt-oss-120b",
#     temperature=0,
# )

# llm = ChatGroq(
#     model="qwen/qwen3.8-27b",
#     temperature=1,
# )

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.1,
)

llm.invoke("hi").content

/home/kishan/Work/Kishan/AI-Engineering-Learning/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text',
  'text': 'Hello! How can I help you today?',
  'extras': {'signature': 'El4KXAFpFH0T41DPS+053B2xzREv8nZ1SUgRMCxupFVnr5ANxkhmdvSnTJK+P4gUi2VaLvTcMjqduEgXCFR/YVRub9mZpCVbv4jBd5RtWiGlNQzWbZs2c1xFaJeF//Uy'}}]

In [20]:

client = MultiServerMCPClient({
    "strapi": {
        "transport": "http",
        "url": "http://localhost:1337/mcp",
        "headers": {
            "Authorization": f"Bearer {os.environ['STRAPI_MCP_TOKEN']}"
        }
    },

    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-filesystem",
            "/home/kishan/Work"
        ],
    },

    "github": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-github",
        ],
        "env": {
            "GITHUB_PERSONAL_ACCESS_TOKEN":
                os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"]
        },
    },
})

# The client returns the tools as normal LangChain tools
mcp_tools = await client.get_tools()

print(f"Loaded {len(mcp_tools)} MCP tools:\n")
for tool in mcp_tools:
    display(Markdown(f"- {tool.name}"))

Loaded 42 MCP tools:



- log

- get_homepage

- read_file

- read_text_file

...

- get_pull_request_files

- get_pull_request_status

- update_pull_request_branch

- get_pull_request_comments

- get_pull_request_reviews

In [21]:

def show_loop(messages):
    for m in messages:
        if getattr(m, "tool_calls", None):
            for call in m.tool_calls:
                print(f"  [model] CALL {call['name']} -> {call['args']}")
        elif getattr(m, "content", None):
            print(f"  [{type(m).__name__}] {str(m.content)[:120]}")


react_agent = create_agent(
    model=llm,
    tools=mcp_tools,
    name="mcp_react_agent",
    system_prompt=(
        "You are an AI assistant. You have MCP tools from a filesystem server "
        "and a GitHub server. Call a tool only when you need live data; "
        "otherwise answer directly from your own knowledge."
    ),
)

result = await react_agent.ainvoke({
    "messages": [("user", "List the contents of the directory /home/kishan/Work.")]
})

show_loop(result["messages"])
print("\nFinal answer:")
print(result["messages"][-1].content)

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '

/home/kishan/Work/Kishan/AI-Engineering-Learning/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema,

  [HumanMessage] List the contents of the directory /home/kishan/Work.
  [model] CALL list_allowed_directories -> {}
  [ToolMessage] [{'type': 'text', 'text': 'Allowed directories:\n/home/kishan/Work', 'id': 'lc_12a7832d-38a2-48b7-b8ad-d27d4e4c903c'}]
  [model] CALL list_directory -> {'path': '/home/kishan/Work'}
  [ToolMessage] [{'type': 'text', 'text': '[DIR] Innvonix\n[DIR] Kishan\n[DIR] Notes\n[DIR] Personal\n[DIR] fodderly_app_api\n[DIR] innv
  [AIMessage] [{'type': 'text', 'text': 'Here are the contents of the directory `/home/kishan/Work`:\n\n- **Innvonix** (Directory)\n- 

Final answer:
[{'type': 'text', 'text': 'Here are the contents of the directory `/home/kishan/Work`:\n\n- **Innvonix** (Directory)\n- **Kishan** (Directory)\n- **Notes** (Directory)\n- **Personal** (Directory)\n- **fodderly_app_api** (Directory)\n- **innvonix-website-fe-2026** (Directory)', 'extras': {'signature': 'El4KXAFpFH0T3vbPW/5fUocCUWqvfGk7epSIYs5PvQD3KporDGdHxIBndCyoimkyapDXqwE7vKt8yA0LnuLXnb4zY9WMqzC

In [22]:
result = await react_agent.ainvoke({
    "messages": [
        ("user", "Search GitHub for repositories matching 'langchain mcp' and show the top 3 results.")
    ]
})

show_loop(result["messages"])
print("\nFinal answer:")
print(result["messages"][-1].content)

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring


Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported

  [HumanMessage] Search GitHub for repositories matching 'langchain mcp' and show the top 3 results.
  [model] CALL search_repositories -> {'query': 'langchain mcp', 'perPage': 3}
  [ToolMessage] [{'type': 'text', 'text': '{\n  "total_count": 3553,\n  "incomplete_results": false,\n  "items": [\n    {\n      "id": 9
  [AIMessage] [{'type': 'text', 'text': "Here are the top 3 GitHub repositories matching **'langchain mcp'**:\n\n1. **[langchain-ai/la

Final answer:
[{'type': 'text', 'text': "Here are the top 3 GitHub repositories matching **'langchain mcp'**:\n\n1. **[langchain-ai/langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)**\n   - **Description:** LangChain 🔌 MCP\n   - **Stars/Activity:** Official LangChain adapters for Model Context Protocol (MCP).\n\n2. **[langchain4j/langchain4j](https://github.com/langchain4j/langchain4j)**\n   - **Description:** LangChain4j is an idiomatic, open-source Java library for building LLM-powered applications on the JVM 

In [24]:

# When no tool is needed the agent skips them and answers directly.
result = await react_agent.ainvoke({
    "messages": [("user", "What is 12 multiplied by 38?")]
})

show_loop(result["messages"])
print("\nFinal answer:")
print(result["messages"][-1].content)

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '

  [HumanMessage] What is 12 multiplied by 38?
  [AIMessage] [{'type': 'text', 'text': '12 multiplied by 38 is **456**.', 'extras': {'signature': 'El4KXAFpFH0T9syRatmzrpEqpgUmqIvqj9

Final answer:
[{'type': 'text', 'text': '12 multiplied by 38 is **456**.', 'extras': {'signature': 'El4KXAFpFH0T9syRatmzrpEqpgUmqIvqj9MW9tOohsPrQwqYtCfjFodR/zkTCGPzgqkpH+7eworuYAA1OboZKq+zNngf0/odGGZY4s9ADS1tTFVAIH8C6lSo10UITMRY'}}]
